# 🔬 Heat Treatment Digital Twin: Post-Training with DPO
**Theme:** Long-Horizon Planning & Instruction Following  
**Objective:** Fine-tune an open-source LLM (Llama-3-8B) using Direct Preference Optimization (DPO) to internalize continuous physical dynamics (thermal inertia/lag) and execute predictive braking in an SMDP environment.

This notebook uses **Unsloth** for highly efficient 4-bit LoRA fine-tuning and **Hugging Face TRL**.

In [ ]:
# %%capture
# !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# !pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
# !pip install datasets

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

In [1]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096 # Supports long-horizon multi-turn recipes
dtype = None # Auto-detect
load_in_4bit = True # Use 4-bit quantization for memory efficiency

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Apply LoRA adapters (Targeting all linear layers for maximum learning capability)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

ModuleNotFoundError: No module named 'unsloth'

In [6]:
from datasets import load_dataset

# Load the synthetic JSONL dataset generated by our physics engine
# Note: Upload 'dpo_dataset.jsonl' to the Colab files pane before running
dataset = load_dataset("json", data_files="dpo_dataset.jsonl", split="train")

def apply_chat_template(example):
    """
    Formats the prompt, chosen, and rejected conversational arrays into 
    the raw string tokens expected by the DPO Trainer using the Llama-3 template.
    """
    prompt_text = tokenizer.apply_chat_template(example["prompt"], tokenize=False, add_generation_prompt=True)
    chosen_text = tokenizer.apply_chat_template(example["chosen"], tokenize=False)
    rejected_text = tokenizer.apply_chat_template(example["rejected"], tokenize=False)
    
    # Strip the BOS token from chosen/rejected to prevent double-tokens when TRL concatenates them
    chosen_text = chosen_text.replace(tokenizer.bos_token, "")
    rejected_text = rejected_text.replace(tokenizer.bos_token, "")
    
    return {
        "prompt": prompt_text,
        "chosen": chosen_text,
        "rejected": rejected_text,
    }

formatted_dataset = dataset.map(apply_chat_template)
print(f"Loaded and formatted {len(formatted_dataset)} DPO preference pairs.")

ImportError: cannot import name 'load_dataset' from 'datasets' (unknown location)

In [3]:
from trl import DPOTrainer, DPOConfig
from transformers import TrainingArguments

# DPO Hyperparameters
dpo_config = DPOConfig(
    output_dir="dpo_heat_treatment_model",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6, # Low learning rate for DPO to prevent catastrophic forgetting
    lr_scheduler_type="cosine",
    max_steps=50, # Set low for hackathon demonstration purposes
    beta=0.1, # The KL penalty factor (how much it can deviate from base model)
    max_length=4096,
    max_prompt_length=2048,
    optim="adamw_8bit",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
)

trainer = DPOTrainer(
    model=model,
    ref_model=None, # Unsloth handles the reference model implicitly via PEFT
    args=dpo_config,
    train_dataset=formatted_dataset,
    tokenizer=tokenizer,
)

# Start training! The model will learn to prefer trajectories with "Predictive Braking"
trainer.train()

ModuleNotFoundError: No module named 'trl'

In [ ]:
# Save the LoRA adapters locally
model.save_pretrained("lora_predictive_braking")
tokenizer.save_pretrained("lora_predictive_braking")

# Optional: Push to Hugging Face Hub
# model.push_to_hub("your-hf-username/llama-3-heat-treatment-dpo", token = "YOUR_HF_TOKEN")